# Stock-take: the chain's order books

Almost four years of orders (January 2023 to September 2026) from five branches. `python generate.py` writes them into `data/`. Find what is wrong before anyone trains on it.

In [ ]:
import pandas as pd

pd.__version__

## 1. Load and look

In [ ]:
orders = pd.read_csv("data/orders.csv")
orders.shape

In [ ]:
orders.head()

In [ ]:
orders.info()

In [ ]:
orders.describe()

## 2. Pick rows and columns

In [ ]:
orders.loc[0:2]

In [ ]:
orders.iloc[0:2]

In [ ]:
orders.loc[0:2, ["dish", "qty"]]

## 3. Hunt the mistakes

In [ ]:
refund = orders["qty"] < 0
refund.sum()

In [ ]:
orders[refund].head(3)

In [ ]:
orders[orders["qty"] > 500]

In [ ]:
orders.isna().sum()

In [ ]:
orders[orders["branch"].isna()].head(3)

In [ ]:
orders["branch"].fillna("Unknown").value_counts()

In [ ]:
orders.duplicated().sum()

In [ ]:
orders["dish"].nunique()   # the menu has 12 dishes

In [ ]:
orders["dish"].value_counts().tail(6)

In [ ]:
menu = pd.read_csv("data/dishes.csv")["dish"]
(~orders["dish"].isin(menu)).sum()

## 4. Draft the clean-up

In [ ]:
clean = orders.drop_duplicates()
clean = clean.dropna(subset=["branch", "qty"])
ok = (clean["qty"] >= 0) & (clean["qty"] <= 500)
clean = clean[ok]
clean["dish"] = clean["dish"].str.strip().str.title()
len(orders) - len(clean)

In [ ]:
clean["dish"].isin(menu).all()

In [ ]:
clean.describe()

In [ ]:
clean.to_parquet("data/orders.parquet", index=False)
pd.read_parquet("data/orders.parquet").dtypes